# Aula 16 · Frequências e a FFT

Esta aula apresenta o [capítulo 16 do site](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/). A ideia central: **todo sinal é uma soma de senos**, e a transformada de Fourier mede quanto de cada frequência ele tem, multiplicando-o por uma "sonda" de cada frequência e somando — a FFT faz isso depressa.

**Ao fim da aula você consegue:**

1. calcular a amplitude de uma frequência com a transformada em laço;
2. calcular e desenhar o espectro de um sinal com a FFT;
3. achar a frequência dominante e os picos de um sinal com ruído;
4. explicar a resolução $1/T$ e o limite $f_s/2$ (aliasing).

**Roteiro:** 🧩 · 1. frequências escondidas · 2. 🧑‍🏫 a transformada · 3. a FFT · 4. um afinador · 5. aliasing · 6. outra área · 🎯 prática · 🧩 o rolamento · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

# --- dados desta aula (baixados do site, se ainda não estiverem aqui) ---
import os
import urllib.request

for ARQUIVO in ["nota_violao.csv", "consumo_energia.csv", "vibracao_motor.csv"]:
    if not os.path.exists(ARQUIVO):
        urllib.request.urlretrieve("https://lacouth.github.io/metodos_telecom-site/dados/" + ARQUIVO, ARQUIVO)
    print(ARQUIVO, "pronto")

## 🧩 O problema da aula

> **Manutenção industrial — o rolamento está gasto?**
>
> *Uma bomba d'água de uma estação de tratamento gira a 1800 rpm (30 voltas por
> segundo). Parar a estação para abrir o motor custa caro; deixar um rolamento
> quebrar custa muito mais. O técnico prendeu um acelerômetro no motor e gravou 2
> segundos de vibração. Pela geometria do rolamento, um defeito na pista externa faz
> o motor vibrar em **107 Hz**. "**Tem 107 Hz nessa vibração, ou é só ruído?**"*

No fim da aula, você procura os picos do espectro da vibração e dá o diagnóstico.

## 1. Frequências escondidas

Um sinal feito de dois senos (5 Hz com amplitude 1 e 12 Hz com amplitude 0,5),
amostrado 100 vezes por segundo, durante 1 s. Para o tempo, `np.linspace` com 100
pontos de 0 a 0,99.

📖 [capítulo 16 · Frequências escondidas](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#frequencias-escondidas)

In [ ]:
# 📦 o sinal — só rode esta célula
fs = 100
N = 100
t = np.linspace(0, (N - 1) / fs, N)
sinal = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)

**✍️ Passo 1.** Desenhe `sinal` contra `t`, com o formato `".-"`.

In [ ]:
# ✍️ passo 1

**Preveja:** dá para ver, no gráfico, que existe um seno de 12 Hz?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Dá para contar as 5 ondas grandes; a de 12 Hz aparece só como uma "ondulação" que
ninguém saberia medir. Com ruído, nem isso.

📖 [capítulo 16 · Frequências escondidas](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#frequencias-escondidas)

</details>

## 2. No quadro: a transformada de Fourier

📖 [capítulo 16 · No quadro: a transformada de Fourier](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#no-quadro-a-transformada-de-fourier)

### 🧑‍🏫 No quadro — a transformada discreta de Fourier

Caderno de papel aberto. No quadro:

1. a "sonda": multiplicar o sinal por um cosseno da frequência $k$ e somar;
2. por que a soma é grande se o sinal tem essa frequência, e quase zero se não tem;
3. o problema do atraso e as duas sondas, cosseno e seno;
4. a amplitude $A_k = \frac{2}{N}\sqrt{C_k^2 + S_k^2}$;
5. as frequências testadas: de $1/T$ em $1/T$, até $f_s/2$.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ C_k = \sum_{n} x_n \cos\!\left(\frac{2\pi k n}{N}\right), \quad
   S_k = \sum_{n} x_n \sin\!\left(\frac{2\pi k n}{N}\right), \quad
   A_k = \frac{2}{N}\sqrt{C_k^2 + S_k^2}. $$

Resolução: $1/T$. Maior frequência: $f_s/2$.

</details>

### 🎯 Sua vez — A sonda

Escreva `amplitude_em(sinal, k)`, que calcula $C_k$ e $S_k$ com um laço sobre as amostras e devolve $A_k$.

In [ ]:
def amplitude_em(sinal, k):
    # sua solução aqui
    pass

In [ ]:
confere(amplitude_em, [
    (([1.0, 0.0, -1.0, 0.0], 1), 1.0),
    ((sinal, 5), 1.0),
    ((sinal, 12), 0.5),
])

<details>
<summary><b>💡 Dica</b></summary>

Dois acumuladores começando em `0.0`; no laço, `sinal[n] * np.cos(2 * np.pi * k * n / N)` e o mesmo com `np.sin`; no fim, `2 * np.sqrt(...) / N`.

</details>

**✍️ Passo 2.** Com a sua `amplitude_em`, calcule a amplitude de `k = 0` até `k = 50` num laço e imprima só as que passarem de 0,1.

In [ ]:
# ✍️ passo 2

**Preveja:** quantas frequências vão aparecer?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Exatamente duas: 5 Hz com amplitude 1,000 e 12 Hz com 0,500. Todas as outras dão
zero, até o arredondamento. Como o sinal tem 1 s, a posição $k$ é a frequência em Hz.

📖 [capítulo 16 · No quadro: a transformada de Fourier](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#no-quadro-a-transformada-de-fourier)

</details>

## 3. A FFT

Os dois laços fazem $N \times N/2$ contas; a FFT faz o mesmo com cerca de
$N \log_2 N$. Para 1 s de áudio de CD, é a diferença entre um bilhão e 700 mil.

📖 [capítulo 16 · A FFT](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#a-fft)

> 🧰 **Comando novo: `np.fft.rfft`**
>
> `np.fft.rfft(sinal)` calcula as somas $C_k$ e $S_k$ de todas as frequências, de 0 até
> $f_s/2$, com a FFT. Cada resultado é um **número complexo**, que guarda as duas somas
> juntas; neste curso, só interessa o tamanho dele (o próximo comando). Com $N$ amostras,
> são `N // 2 + 1` resultados.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
espectro = np.fft.rfft([1.0, 0.0, -1.0, 0.0])
print(len(espectro))

> 🧰 **Comando novo: `np.abs`**
>
> `np.abs(a)` dá o valor absoluto de cada elemento. Num resultado da FFT, dá o
> **tamanho** $\sqrt{C_k^2 + S_k^2}$; a amplitude é `2 * np.abs(espectro) / N`.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
print(np.abs(np.array([-2, 3])))
print(2 * np.abs(np.fft.rfft([1.0, 0.0, -1.0, 0.0])) / 4)

> 🧰 **Comando novo: `np.fft.rfftfreq`**
>
> `np.fft.rfftfreq(N, 1 / fs)` dá a frequência, em Hz, de cada posição do resultado da
> `rfft`. O segundo argumento é o **tempo entre amostras**, `1 / fs`.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
print(np.fft.rfftfreq(4, 1 / 4))

**✍️ Passo 3.** Calcule o espectro de `sinal` com a FFT (amplitudes e frequências) e desenhe amplitude contra frequência, com `"o-"`.

In [ ]:
# ✍️ passo 3

**Preveja:** o gráfico vai ter os mesmos dois picos do passo 2?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Os mesmos: 1,000 em 5 Hz e 0,500 em 12 Hz, 51 frequências de 0 a 50 Hz. A FFT não
é outra conta, é a mesma conta feita de um jeito esperto.

📖 [capítulo 16 · A FFT](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#a-fft)

</details>

## 4. Um afinador

A quinta corda do violão deve soar em **110 Hz**. O arquivo tem meio segundo dela, a
8000 amostras por segundo, com o barulho da sala.

📖 [capítulo 16 · Um afinador](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#um-afinador)

In [ ]:
# 📦 dados prontos — só rode esta célula
dados = np.loadtxt("nota_violao.csv", delimiter=",", skiprows=1)
t_som = dados[:, 0]
som = dados[:, 1]
fs_som = 8000

### 🎯 Sua vez — A frequência dominante

Escreva `frequencia_dominante(sinal, fs)`: tire a média do sinal, calcule a FFT e as frequências e devolva a frequência do maior pico.

In [ ]:
def frequencia_dominante(sinal, fs):
    # sua solução aqui
    pass

In [ ]:
confere(frequencia_dominante, [
    ((sinal, 100), 5.0),
    ((sinal + 3, 100), 5.0),
    ((som, fs_som), 108.0),
])

<details>
<summary><b>💡 Dica</b></summary>

`sinal - np.mean(sinal)`, depois `np.abs(np.fft.rfft(...))`, `np.fft.rfftfreq(len(sinal), 1 / fs)` e a frequência na posição do `np.argmax`.

</details>

**✍️ Passo 4.** Desenhe as 400 primeiras amostras do som no tempo e, numa segunda figura, as 400 primeiras frequências do espectro. Imprima a frequência dominante e a resolução (`freq[1]`).

In [ ]:
# ✍️ passo 4

**Preveja:** a corda está afinada, alta ou baixa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

No tempo, só ruído. No espectro, um pico claro em **108 Hz** e os
harmônicos em cerca de 217, 325 e 434 Hz. A corda está **baixa**. A resolução é de
2 Hz (meio segundo de som), então o afinador não distingue 108 de 108,5 Hz: para
isso, seria preciso gravar mais tempo.

📖 [capítulo 16 · Um afinador](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#um-afinador)

</details>

## 5. Amostrar rápido o bastante

📖 [capítulo 16 · Amostrar rápido o bastante](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#amostrar-rapido-o-bastante)

**✍️ Passo 5.** Amostre um seno de **12 Hz** só **15 vezes por segundo**, durante 1 s (`np.linspace(0, 14 / 15, 15)`), e use a sua `frequencia_dominante`.

In [ ]:
# ✍️ passo 5

**Preveja:** a FFT vai dizer 12 Hz?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Vai dizer **3 Hz**. As 15 amostras caem exatamente sobre um seno de 3 Hz, e nenhum
método tem como saber a diferença: é o **aliasing**. Só se enxergam frequências
até $f_s/2 = 7{,}5$ Hz.

📖 [capítulo 16 · Amostrar rápido o bastante](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#amostrar-rapido-o-bastante)

</details>

> ⚠️ **Armadilha.** Nenhuma conta depois da gravação conserta o aliasing: a informação já se perdeu.
Por isso os conversores filtram o sinal **antes** de amostrar.

## 6. Mesmo método, outra área

**Setor elétrico.** No capítulo 15, os ciclos de 24 h e 168 h do consumo de energia
eram conhecidos de antemão. A FFT os descobre sozinha. Com uma amostra por hora, as
frequências saem em ciclos por hora (`np.fft.rfftfreq(N, 1)`), e o período é $1/f$.

📖 [capítulo 16 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#mesmo-metodo-outra-area)

In [ ]:
# 📦 dados prontos — só rode esta célula
consumo = np.loadtxt("consumo_energia.csv", delimiter=",", skiprows=1)[:, 1]

**✍️ Passo 6.** Com `frequencia_dominante(consumo, 1)`, ache a frequência mais forte do consumo e imprima o período `1 / f`.

In [ ]:
# ✍️ passo 6

**Preveja:** qual período vai sair: o do dia ou o da semana?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**24 horas**: o ciclo diário é o mais forte (5,7 MW de amplitude). O semanal
(168 h) e um "harmônico" de 12 h vêm depois; o de 12 h existe porque o dia típico
não é um seno perfeito.

📖 [capítulo 16 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-fft/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

A prática desta aula é o problema, logo abaixo: achar **todos** os picos de uma vibração.

## 🧩 Resolvendo o problema

> *"**Tem 107 Hz nessa vibração, ou é só ruído?**"* — o técnico de manutenção.

In [ ]:
# 📦 dados prontos — só rode esta célula
dados = np.loadtxt("vibracao_motor.csv", delimiter=",", skiprows=1)
t_motor = dados[:, 0]
vibracao = dados[:, 1]      # aceleração, em g
fs_motor = 2000             # amostras por segundo, durante 2 s

### 🎯 Sua vez — Os picos do espectro

Escreva `picos(sinal, fs, limiar)`, que tira a média do sinal, calcula as amplitudes
(`2 * np.abs(...) / N`) e as frequências, e devolve a **lista** das frequências cuja
amplitude passa de `limiar`.

In [ ]:
def picos(sinal, fs, limiar):
    # sua solução aqui
    pass

In [ ]:
confere(picos, [
    ((sinal, 100, 0.1), [5.0, 12.0]),
    ((sinal, 100, 0.7), [5.0]),
    ((vibracao, fs_motor, 0.15), [30.0, 60.0, 107.0]),
])

<details>
<summary><b>💡 Dica</b></summary>

Monte `achados = []` e, num laço pelas posições `k`, faça `achados.append(freq[k])` quando `amplitude[k] > limiar`.

</details>

In [ ]:
resposta = picos(vibracao, fs_motor, 0.15)
if resposta is not None:
    print("picos acima de 0,15 g:", resposta)
    for f in resposta:
        print(f"  {f:6.1f} Hz = {f / 30:.2f} vezes a rotação")

<details>
<summary><b>▶ O diagnóstico</b></summary>

Três picos: **30 Hz** (a própria rotação, que todo motor tem), **60 Hz** (o dobro da
rotação, sinal típico de um leve desalinhamento do eixo) e **107 Hz**, que não é um
múltiplo inteiro da rotação: é a frequência de defeito da pista externa do rolamento.
No gráfico do tempo, ele é invisível (amplitude de 0.25 g, com um ruído
de quase 1 g). No espectro, ele se destaca: fora dos três picos, nenhuma frequência
passa de 0.09 g.

Diagnóstico: **o rolamento está começando a falhar**. A troca pode ser agendada para a
próxima parada programada, e a vibração, medida de novo toda semana. Se o pico de
107 Hz crescer, a troca é antecipada. Isso é a **manutenção preditiva**: consertar
antes de quebrar, sem abrir o motor à toa.

Uma escolha honesta: o limiar de 0,15 g foi escolhido olhando o espectro. Na prática,
as empresas definem os limites por norma e pelo histórico de cada máquina.

</details>

## 📋 A lista

Abra a [Lista 16](https://lacouth.github.io/metodos_telecom-site/listas/lista16/). O **Exercício 01** é à mão (✏️): uma transformada de 4
amostras, a resolução e o aliasing. Comece por ele, no papel.

**a)** Para $x = [1, 0, -1, 0]$, quanto vale $C_0 = \sum_n x_n \cos(0)$?

<details>
<summary><b>▶ Resposta</b></summary>

$1 + 0 - 1 + 0 = 0$: com $k = 0$, a sonda é a constante 1, e $C_0$ é a soma do sinal (a média vezes $N$).

</details>

Termine o exercício e siga para o **Exercício 02**, a sonda como função.

## 🚪 Antes de sair

**1.** Por que a soma $\sum x_n \cos(2\pi k n/N)$ fica grande quando o sinal tem a frequência $k$?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque o sinal e o cosseno sobem e descem juntos: os produtos são quase todos positivos e se acumulam. Numa frequência que o sinal não tem, os produtos se cancelam.

</details>

**2.** Um sensor grava 4 s a 500 amostras por segundo. Qual a resolução? E a maior frequência visível?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Resolução $1/T = 0{,}25$ Hz; maior frequência $f_s/2 = 250$ Hz.

</details>

**3.** Por que tirar a média do sinal antes de procurar o pico?

<details>
<summary><b>▶ Resposta da 3</b></summary>

A média é a frequência zero; com uma média grande, o pico em 0 Hz ganharia de todos os outros e esconderia o que interessa.

</details>

## 🏠 Para casa

- Termine a [Lista 16](https://lacouth.github.io/metodos_telecom-site/listas/lista16/).
- Leia os [temas do projeto final](https://lacouth.github.io/metodos_telecom-site/projeto/temas/): o projeto usa a regressão ou
  as séries temporais, e a FFT pode entrar nele.